The purpose of this notebook is to create a training data set based on unprocessed Earth Engine imagery in the Earth Engine landing zone. The preprocessing steps are:
1. Obtain filtered list of file names based on image attributes from metadata database
2. Generate training data tifs

To-do (1 hour):
- Finish pulling the data (1 min) - done
- Idenitfy what the right thermal band is for landsat 8 (20 min)
- Update SQL with new filters and summer months (30 min) - done
- Rerun everything (20 min) - done


In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.expanduser("~"),"Desktop","projects", "GlacierView",
                                "src","segmentation","helpers"))
import read, preprocess, explore
from tqdm import tqdm

import rasterio
import pandas as pd

import pickle

import numpy as np
import tifffile
import geopandas as gpd
from datetime import date
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from PIL import Image
from scipy.stats import logistic
import cv2

import importlib
importlib.reload(read)
importlib.reload(preprocess)
importlib.reload(explore)

# TRAINING

In [ ]:
#inputs
data_label = "localized_time_series_for_training_c02_t1_l2"
dem_data_label = "localized_time_series_for_training_c02_t1_l2"
dem_label = "NASADEM"
glacier_view_dir = os.path.join(os.path.expanduser('~'),"Desktop","projects","GlacierView")
glaciers_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","ee_landing_zone",data_label, "landsat")
dems_dir = os.path.join(glacier_view_dir,"src","earth_engine", "data","ee_landing_zone",dem_data_label, "dems")
masks_dir = os.path.join(glacier_view_dir, "src","segmentation","training","data","masks_staging_2")
log_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","ee_landing_zone",data_label, "logs")
glims_ids = sorted([f for f in os.listdir(glaciers_dir) if not f.startswith('.')])

#outputs
processed_training_data = os.path.join(glacier_view_dir, "src","segmentation","training","data","processed_training_data_summer_months")
images_write_path = os.path.join(processed_training_data, "images")
masks_write_path = os.path.join(processed_training_data, "masks")

In [ ]:
common_bands = ['blue','green','red','nir','swir','thermal']
dim = (128,128)

In [ ]:
metadata_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","processed_metadata",data_label)
df = pd.read_csv(os.path.join(metadata_dir,"filtered_training_data_summer_months.csv"))

In [ ]:
for idx, row in tqdm(df.iterrows()):
    images = {}
    dems = {}
    masks = {}

    # if row.file_name.split("_")[2] != "L8":
    #     continue
        
    images[row.glims_id] = read.get_rasters(os.path.join(glaciers_dir,row.glims_id),row.file_name )
    dems[row.glims_id] = read.get_dem(os.path.join(dems_dir,row.glims_id + '_' + dem_label + '.tif'))
    
    images[row.glims_id] = preprocess.get_common_bands(images[row.glims_id],common_bands)
    images[row.glims_id] = preprocess.normalize_rasters(images[row.glims_id])
    images[row.glims_id] = preprocess.resize_rasters(images[row.glims_id],dim)

    # plt.title("Before: " + row.file_name)
    # plt.imshow(dems[row.glims_id][f"{row.glims_id}_NASADEM.tif"][:,:,0])
    # plt.show()
    
    dems[row.glims_id] = preprocess.normalize_rasters(dems[row.glims_id])
    dems[row.glims_id] = preprocess.resize_rasters(dems[row.glims_id], dim)

    mask_file_name = f"{row.glims_id}.tif"
    try:
        img = Image.open(os.path.join(masks_dir, mask_file_name))
    except FileNotFoundError:
        continue
    masks[row.glims_id] = {mask_file_name: np.expand_dims(np.array(img),2)}
    masks[row.glims_id] = preprocess.resize_rasters(masks[row.glims_id], dim)

    combined_to_stack = []

    image = images[row.glims_id]
    dem = dems[row.glims_id]
    mask = masks[row.glims_id]

    # plt.title("After: " + row.file_name)
    # plt.imshow(dem[f"{row.glims_id}_NASADEM.tif"][:,:,0])
    # plt.show()


    X = [np.concatenate((image[file_name], dem[f"{row.glims_id}_NASADEM.tif"]),axis = 2) for file_name in image]
#     if np.sum(smoothed_image == 0) < 50000: #convert to percent
#         combined_to_stack.append(smoothed_image)
    X = np.stack(X)
    tifffile.imsave(os.path.join(images_write_path,f"{row.file_name}"), X, planarconfig='contig')
    tifffile.imsave(os.path.join(masks_write_path,f"{row.glims_id}.tif"),mask[f'{row.glims_id}.tif'])    
    

## Inspect training data

### Step 1 read the images and masks

In [ ]:
import random

image_file_names = os.listdir(images_write_path)
image_file_names = [file_name for file_name in image_file_names if file_name.endswith('.tif')]
mask_file_names = os.listdir(masks_write_path)
mask_file_names = [file_name for file_name in mask_file_names if file_name.endswith('.tif')]

random.shuffle(image_file_names)
random.shuffle(mask_file_names)

### Step 2 combine the images and masks

In [ ]:
written_training_images = {}
for file_name in tqdm(image_file_names):
    device = file_name.split("_")[2]
    # if device != "L8":
    #     continue
    glims_id = file_name.split("_")[0]
    with rasterio.open(os.path.join(images_write_path,file_name)) as src:
        raster = src.read()
    with rasterio.open(os.path.join(masks_write_path, glims_id + ".tif",)) as src:
        mask = src.read()
    combined = np.concatenate((raster, mask), axis = 0)
    combined_and_rolled = np.rollaxis(combined, 0,3)
    written_training_images[file_name] = combined_and_rolled

### Step 3 plot the images and masks

In [ ]:
for file_name, written_training_image in written_training_images.items():
    counter = 0
    band_labels = ['blue','green','red','nir','swir','thermal', 'dem','mask']
    device = file_name.split("_")[2]
    # if device != "L8":
    #     continue
    fig, ax = plt.subplots(2,4, figsize = (10,5))
    fig.suptitle(file_name, fontsize=16)
    for i in range(2):
        for j in range(4):
            ax[i,j].imshow(written_training_image[:,:,counter])
            ax[i,j].set_title(band_labels[counter])
            counter += 1
    plt.show()